In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb spacy tqdm
!pip install --upgrade contractions

In [2]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map
import random

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login()  # reads WANDB_API_KEY from the environment  # Replace with your actual API key
wandb.init(project="Spring2025", reinit=True)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dhruvjoshi892000 (dhruvjoshi892000-pace-university). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Tracking run with wandb version 0.19.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20250404_082109-ygj582kd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fluent-morning-68
wandb: ⭐️ View project at https://wandb.ai/dhruvjoshi892000-pace-university/Spring2025
wandb: 🚀 View run at https://wandb.ai/dhruvjoshi892000-pace-university/Spring2025/runs/ygj582kd


In [3]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

Using cuda for training


In [4]:
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [5]:
import spacy

# Load spaCy model with GPU support
spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available

    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/15_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "nlpaueb/legal-bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)



spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [31:50<00:00,  4.41it/s]


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [6]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [7]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=279).to(device)

# Adjusted LoRA Configuration for Long-Form Text Classification
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.4,
    target_modules=[
        "attention.self.query",
        "attention.self.key",
        "attention.self.value",
        "attention.output.dense",
        "intermediate.dense",
        "output.dense"
    ],
    bias="none",
    inference_mode=False
)

# Integrate LoRA with the model
model = get_peft_model(model, lora_config).to(device)
model.print_trainable_parameters()


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,868,759 || all params: 112,565,550 || trainable%: 2.5485


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [9]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/results",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="/kaggle/working/logs",
#     label_names=["labels"],
#     logging_steps=10,
#     num_train_epochs=20,  # More epochs for better legal text learning
#     per_device_train_batch_size=4,  # Lower batch size for stability
#     per_device_eval_batch_size=4,
#     learning_rate=2e-5,  # Reduce LR for better fine-tuning
#     warmup_ratio=0.1,  # Stabilize early training
#     weight_decay=0.01,
#     metric_for_best_model="balanced_accuracy",
#     load_best_model_at_end=True,
#     report_to=["wandb"],  # Enable W&B logging
#     fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
# )

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",        # Save the model every epoch
    learning_rate=2e-5,           # Reduced LR for stability
    per_device_train_batch_size=16,  # Increase if memory allows
    per_device_eval_batch_size=16,
    num_train_epochs=20,         # Keep training epochs
    weight_decay=0.05,           # Regularization to prevent overfitting
    logging_dir="./logs",
    logging_steps=50,            # Log progress every 50 steps
    load_best_model_at_end=True, # Load the best checkpoint
    metric_for_best_model="eval_loss",  # Select the best model based on validation loss
    lr_scheduler_type="cosine",  # Cosine decay learning rate
    warmup_ratio=0.1,            # 10% warmup steps
    gradient_accumulation_steps=2,  # Stabilizes training for smaller batch sizes
    fp16=True,                   # Mixed precision training
    report_to="wandb",           # Log results to Weights & Biases
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

print("started training.")
# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


started training.


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,4.985500,4.097600,0.314727,0.104314,0.175787,0.314727,0.207131
2,2.365000,2.181621,0.370546,0.119162,0.161223,0.370546,0.224547
3,1.815900,1.768242,0.457245,0.156314,0.306172,0.457245,0.356598
4,1.624800,1.532218,0.519002,0.188723,0.403859,0.519002,0.408060
5,1.443000,1.361559,0.621140,0.320401,0.562086,0.621140,0.552428
6,1.273900,1.227713,0.666271,0.361968,0.594172,0.666271,0.615962
7,1.218100,1.121885,0.697150,0.390851,0.611381,0.697150,0.645188
8,1.151700,1.051871,0.710214,0.396199,0.631181,0.710214,0.665008
9,1.060500,1.004553,0.714964,0.401302,0.626491,0.714964,0.665107
10,0.954300,0.965960,0.720903,0.409324,0.680161,0.720903,0.672194


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'eval_loss': 0.8576974272727966, 'eval_accuracy': 0.7482185273159145, 'eval_balanced_accuracy': 0.4546695786390646, 'eval_precision': 0.7051023023272872, 'eval_recall': 0.7482185273159145, 'eval_f1': 0.7153793167931058, 'eval_runtime': 17.5074, 'eval_samples_per_second': 48.094, 'eval_steps_per_second': 3.027, 'epoch': 20.0}
